# Этап 5: ResNet50-IBN-a + GeM + BNNeck

Ноутбук выполняет небольшой screening learning rate на внутреннем identity-disjoint split, затем обучает выбранную конфигурацию с тремя seed и сравнивает средние метрики с активной OSNet. Веса MVP автоматически не заменяются. Достаточно выполнить **Run All**.

In [1]:
import os
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'training').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'training').is_dir():
    raise RuntimeError('Запустите notebook из репозитория Car-classification-MSK')
os.chdir(ROOT)
print('Repository:', ROOT)

Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK


In [2]:
import json
import torch

from training.pipeline import ensure_splits, select_device
from training.resnet_ibn import EMBEDDING_DIMENSION, ResNetIBNReIDModel
from training.stage5 import (export_stage5_winner, run_backbone_screening,
                             run_backbone_seeds)

VARIANT_DIR = ROOT / 'ResNet50-IBN' / 'variant_05_gem_bnneck'
RESULTS_DIR = VARIANT_DIR / 'results'
WEIGHTS_DIR = VARIANT_DIR / 'weights'
BASE_CONFIG = (ROOT / 'OSNet-AIN-x1.0' / 'variant_02_hpo_bnneck_supcon' /
               'results' / 'selected_config.json')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
device = select_device()
if device.type == 'cpu':
    raise RuntimeError(
        'MPS/CUDA недоступен. Не запускаем долгий ResNet50 на CPU; '
        'проверьте окружение PyTorch и перезапустите kernel.'
    )
rows, split = ensure_splits()
print('Device:', device)
print('Train identities:', len(split['identities']['train']))
print('Embedding dimension:', EMBEDDING_DIMENSION)

Device: mps
Train identities: 925
Embedding dimension: 2048


## Smoke test
Проверяет локальную архитектуру без скачивания весов и без обучения.

In [3]:
smoke = ResNetIBNReIDModel(num_classes=3).eval().to(device)
with torch.inference_mode():
    logits, raw, embedding = smoke(torch.zeros(2, 3, 64, 64, device=device))
assert logits.shape == (2, 3)
assert raw.shape == embedding.shape == (2, EMBEDDING_DIMENSION)
del smoke, logits, raw, embedding
if device.type == 'mps':
    torch.mps.empty_cache()
elif device.type == 'cuda':
    torch.cuda.empty_cache()
print('Smoke test: OK')

Smoke test: OK


## 1. Внутренний screening
Три learning rate сравниваются по raw mAP@10 только внутри 925 train identity. Outer calibration и validation здесь не используются. Первый запуск скачает официальный ImageNet checkpoint IBN-Net v1.0; последующие используют локальный кэш PyTorch.

In [4]:
screening = run_backbone_screening(
    rows, split, device, BASE_CONFIG, RESULTS_DIR, WEIGHTS_DIR,
    learning_rates=(3e-5, 6e-5, 1e-4), epochs=6,
)
print(json.dumps({
    'winner': screening['winner']['name'],
    'best_epoch': screening['winner']['best_epoch'],
    'best_inner_mAP': screening['winner']['best_mAP'],
}, ensure_ascii=False, indent=2))

epoch 1:   0%|          | 0/46 [00:00<?, ?it/s]

Epoch 1/6 | осталось эпох: 5 | эпоха: 00:00:51 | прошло: 00:00:51 | ETA: 00:04:16 | best mAP: 0.1461


epoch 2:   0%|          | 0/46 [00:00<?, ?it/s]

Epoch 2/6 | осталось эпох: 4 | эпоха: 00:00:52 | прошло: 00:01:44 | ETA: 00:03:28 | best mAP: 0.1865


epoch 3:   0%|          | 0/46 [00:00<?, ?it/s]

...

Epoch 4/6 | осталось эпох: 2 | эпоха: 00:00:51 | прошло: 00:03:30 | ETA: 00:01:45 | best mAP: 0.2772


epoch 5:   0%|          | 0/46 [00:00<?, ?it/s]

Epoch 5/6 | осталось эпох: 1 | эпоха: 00:00:52 | прошло: 00:04:22 | ETA: 00:00:52 | best mAP: 0.2993


epoch 6:   0%|          | 0/46 [00:00<?, ?it/s]

Epoch 6/6 | осталось эпох: 0 | эпоха: 00:00:51 | прошло: 00:05:14 | ETA: 00:00:00 | best mAP: 0.2993
{
  "winner": "lr_1e-04",
  "best_epoch": 5,
  "best_inner_mAP": 0.29925916350023496
}


## 2. Три seed
Выбранная конфигурация заново стартует из одинаковых официальных ImageNet-весов. Число эпох фиксировано по inner screening. Порог отказа и параметры reranking выбираются на calibration; validation каждого seed оценивается один раз. Все запуски продолжаются после прерывания.

In [5]:
comparison = run_backbone_seeds(
    rows, split, screening, device, RESULTS_DIR, WEIGHTS_DIR,
)
print(json.dumps({
    'aggregate': comparison['aggregate'],
    'beats_active_reference': comparison['beats_active_reference'],
    'selected_representative': comparison['selected_representative'],
}, ensure_ascii=False, indent=2))

epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 1/5 | осталось эпох: 4 | эпоха: 00:00:52 | прошло: 00:00:52 | ETA: 00:03:27


epoch 2:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 2/5 | осталось эпох: 3 | эпоха: 00:00:51 | прошло: 00:01:43 | ETA: 00:02:34


epoch 3:   0%|          | 0/57 [00:00<?, ?it/s]

...

Epoch 3/5 | осталось эпох: 2 | эпоха: 00:00:51 | прошло: 00:02:34 | ETA: 00:01:43


epoch 4:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 4/5 | осталось эпох: 1 | эпоха: 00:00:51 | прошло: 00:03:26 | ETA: 00:00:51


epoch 5:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 5/5 | осталось эпох: 0 | эпоха: 00:00:51 | прошло: 00:04:17 | ETA: 00:00:00
Calibration reranking: k1=10, k2=1
Calibration reranking: k1=10, k2=3
Calibration reranking: k1=10, k2=6
Calibration reranking: k1=20, k2=1
Calibration reranking: k1=20, k2=3
Calibration reranking: k1=20, k2=6
Calibration reranking: k1=30, k2=1
Calibration reranking: k1=30, k2=3
Calibration reranking: k1=30, k2=6
{
  "aggregate": {
    "mean_mAP_at_10": 0.27297071407660384,
    "std_mAP_at_10": 0.01732393128533559,
    "mean_candidate_score": 0.4004566646329321,
    "std_candidate_score": 0.04353868591949014,
    "mean_quality_score": 0.16288248779776493,
    "std_quality_score": 0.012009888612048748
  },
  "beats_active_reference": false,
  "selected_representative": {
    "name": "seed_20260915",
    "seed": 20260915
  }
}


## 3. Экспорт и benchmark
Экспортируется репрезентативный seed, проверяется PyTorch/ONNX parity, размер модели и batch=1 latency. ONNX остаётся экспериментальным и не подключается к backend автоматически.

In [6]:
export = export_stage5_winner(
    comparison, split, rows, WEIGHTS_DIR,
    WEIGHTS_DIR / 'resnet50_ibn_stage5.onnx',
    RESULTS_DIR / 'export_summary.json',
    benchmark_device=device,
)
print(json.dumps({
    'onnx': export['onnx'],
    'onnx_size_mb': export['onnx_size_mb'],
    'embedding_dimension': export['embedding_dimension'],
    'parity': export['parity'],
    'benchmark': export['benchmark'],
}, ensure_ascii=False, indent=2))

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


{
  "onnx": "/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/ResNet50-IBN/variant_05_gem_bnneck/weights/resnet50_ibn_stage5.onnx",
  "onnx_size_mb": 89.66443729400635,
  "embedding_dimension": 2048,
  "parity": {
    "max_absolute_difference": 7.62939453125e-06,
    "cosine_similarity": 0.9999998211860657
  },
  "benchmark": {
    "pytorch": {
      "device": "mps",
      "median_ms": 6.1886040202807635,
      "p95_ms": 6.508735747775062,
      "samples": 30,
      "includes": "model forward only, batch=1"
    },
    "onnx_cpu": {
      "device": "ONNX Runtime CPU",
      "median_ms": 11.848271009512246,
      "p95_ms": 11.967485584318638,
      "samples": 30,
      "includes": "model forward only, batch=1"
    }
  }
}


## Интерпретация
Если `beats_active_reference=true`, результат можно отдельно проверить в MVP. Если `false`, активная OSNet остаётся без изменений. Сам по себе лучший одиночный seed не считается достаточным основанием для замены модели.